# Transformer-Based Next-Token Prediction

**Focus:** Self-attention and language modelling

## Objective
Build a small decoder-only Transformer in PyTorch and train it to predict the next character in a text sequence. This compact experiment demonstrates autoregressive language modelling without a pretrained model or paid API.

## Learning goals
- Understand tokenization, embeddings, positional information, and causal self-attention.
- Train a Transformer using next-token prediction and cross-entropy loss.
- Generate text one token at a time and evaluate loss on held-out text.

**Dataset:** A small, original, built-in text corpus. This keeps the notebook self-contained and runnable without downloading files. The model predicts **characters**, not words or subword tokens, so this is an educational demonstration rather than a general-purpose language model.

**Requirements:** Python 3.10+ and PyTorch. PyTorch is usually preinstalled in Google Colab.

## 1. Imports and reproducibility

Set random seeds to make the experiment easier to reproduce. Results may still vary slightly across hardware and library versions.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")

## 2. Prepare the text corpus

The short, original corpus is repeated to create enough training windows for a small demonstration. Characters are mapped to integer IDs. The final portion of the sequence is held out for validation.

In [ ]:
corpus = """
A small language model learns patterns from examples.
A transformer uses attention to connect information across a sequence.
The next token depends on the tokens that came before it.
Training adjusts model parameters to reduce prediction error.
Attention helps each position gather useful context from earlier positions.
A language model does not understand the world simply because it predicts text.
Its output reflects the data, architecture, and training process.
Clear experiments make it easier to study how neural networks learn.
""".strip().lower()

text = (corpus + "\n") * 40
vocab = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for ch, i in stoi.items()}
encoded = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

split_index = int(0.9 * len(encoded))
train_data = encoded[:split_index]
val_data = encoded[split_index:]

print(f"Characters: {len(encoded):,}")
print(f"Vocabulary size: {len(vocab)}")
print(f"Training characters: {len(train_data):,}")
print(f"Validation characters: {len(val_data):,}")

## 3. Create next-token training batches

Each input window contains a sequence of characters. Its target is the same sequence shifted one character to the right, so the model learns to predict the next character at every position.

In [ ]:
BLOCK_SIZE = 64
BATCH_SIZE = 32

def get_batch(split: str):
    data = train_data if split == "train" else val_data
    if len(data) <= BLOCK_SIZE + 1:
        raise ValueError("Dataset must contain more characters than the context window.")
    starts = torch.randint(0, len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in starts])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("Input batch shape:", tuple(xb.shape))
print("Target batch shape:", tuple(yb.shape))
print("Input example: ", ''.join(itos[i.item()] for i in xb[0][:60]))
print("Target example:", ''.join(itos[i.item()] for i in yb[0][:60]))

## 4. Build a decoder-only Transformer

The model combines token embeddings with positional embeddings, then applies causal self-attention and a feed-forward network. A causal mask prevents a position from using future characters. The final linear layer returns a score for each possible next character.

Scaled dot-product attention is:

\[
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V
\]

Here, \(Q\), \(K\), and \(V\) are queries, keys, and values; \(M\) masks future positions.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        if n_embd % n_head != 0:
            raise ValueError("n_embd must be divisible by n_head.")
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("causal_mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        batch, seq_len, channels = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self.causal_mask[:, :, :seq_len, :seq_len]
        scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, channels)
        return self.resid_dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class TinyTransformerLM(nn.Module):
    def __init__(
        self, vocab_size: int, block_size: int, n_embd: int = 96,
        n_head: int = 4, n_layer: int = 2, dropout: float = 0.1
    ):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            TransformerBlock(n_embd, n_head, block_size, dropout)
            for _ in range(n_layer)
        ])
        self.final_ln = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        _, seq_len = idx.shape
        if seq_len > self.block_size:
            raise ValueError(f"Sequence length {seq_len} exceeds block size {self.block_size}.")
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)[None, :, :]
        x = self.blocks(x)
        logits = self.lm_head(self.final_ln(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 0.8):
        if temperature <= 0:
            raise ValueError("temperature must be greater than zero.")
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx


model = TinyTransformerLM(vocab_size=len(vocab), block_size=BLOCK_SIZE).to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {parameter_count:,}")
logits, initial_loss = model(xb, yb)
print("Logits shape:", tuple(logits.shape))
print(f"Initial batch loss: {initial_loss.item():.4f}")

## 5. Train the model

AdamW updates the model parameters using cross-entropy loss. Validation loss is measured periodically but is not used to update the weights. The training duration is deliberately short so the notebook remains practical in Colab.

In [ ]:
MAX_STEPS = 800
EVAL_INTERVAL = 100
EVAL_BATCHES = 20
LEARNING_RATE = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
train_steps, train_losses, val_losses = [], [], []

@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(EVAL_BATCHES):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses.append(loss.item())
        results[split] = sum(losses) / len(losses)
    model.train()
    return results

model.train()
for step in range(1, MAX_STEPS + 1):
    x, y = get_batch("train")
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step == 1 or step % EVAL_INTERVAL == 0:
        metrics = estimate_loss()
        train_steps.append(step)
        train_losses.append(metrics["train"])
        val_losses.append(metrics["val"])
        print(
            f"Step {step:4d}/{MAX_STEPS} | "
            f"train loss: {metrics['train']:.4f} | "
            f"validation loss: {metrics['val']:.4f}"
        )

## 6. Inspect the learning curves

Cross-entropy loss measures how well the model assigns probability to the correct next character; lower is better. If training loss falls while validation loss rises, that can indicate overfitting.

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(train_steps, train_losses, marker="o", label="Training loss")
plt.plot(train_steps, val_losses, marker="o", label="Validation loss")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("Transformer training and validation loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Evaluate on held-out text

Report the final validation cross-entropy loss and perplexity. Perplexity is the exponential of cross-entropy loss and reflects the model's uncertainty about the next character. Since validation text comes from the same small repeated corpus, this is a demonstration metric, not a benchmark of general language ability.

In [ ]:
final_metrics = estimate_loss()
val_loss = final_metrics["val"]
perplexity = math.exp(val_loss) if val_loss < 20 else float("inf")

print(f"Final training loss:   {final_metrics['train']:.4f}")
print(f"Final validation loss: {val_loss:.4f}")
print(f"Validation perplexity: {perplexity:.2f}")

## 8. Generate text

The model samples one character at a time from its predicted distribution. The generated text may be repetitive or grammatically incorrect because the model and corpus are intentionally small.

In [ ]:
prompt = "the "
unknown_chars = sorted(set(prompt) - set(stoi))
if unknown_chars:
    raise ValueError(f"Prompt contains characters outside the vocabulary: {unknown_chars}")

context = torch.tensor([[stoi[ch] for ch in prompt]], dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=300, temperature=0.8)[0].tolist()
print(''.join(itos[i] for i in generated))

## 9. Inspect next-character predictions

For a short context, display the most probable next characters and their probabilities. These are model predictions, not guaranteed outcomes.

In [ ]:
context_text = "attention "
context_ids = torch.tensor([[stoi[ch] for ch in context_text]], dtype=torch.long, device=device)
model.eval()
with torch.no_grad():
    next_logits, _ = model(context_ids)
    next_probs = F.softmax(next_logits[0, -1, :], dim=-1)
    top_probs, top_ids = torch.topk(next_probs, k=min(8, len(vocab)))

print(f"Context: {context_text!r}")
print("Top next-character predictions:")
for prob, idx in zip(top_probs.tolist(), top_ids.tolist()):
    print(f"  {itos[idx]!r:8s} {prob:.3f}")

## Conclusion

This notebook implements a compact decoder-only Transformer with causal self-attention and trains it for character-level next-token prediction. It demonstrates tokenization, shifted targets, attention masking, optimization, validation loss, perplexity, and autoregressive generation.

The experiment uses a small repeated toy corpus. Its metrics should not be compared directly with modern language models, and generated text is not expected to be consistently coherent. Possible extensions include a larger text dataset, subword tokenization, more Transformer layers, learning-rate scheduling, and comparison with a recurrent neural network baseline.